## 캐글 노트북에서 실험한 DDP 간단 구현 
- 학습 방법 : Pytorch 공식문서 튜토리얼 뼈대 코드에 Gemini의 가이드 학습을 통해 필수 부분만 전달받은 후 공식 문서 보고 구현
- dist.braodcast(), dist.all_reduce()를 통해 구현
- 미분의 선형성을 이용
- 데이터 Sampler는 생략

In [6]:
%%writefile train.py


# 언제 값을 합칠지, 어떤 값을 합칠지, 평균은 어떻게 낼 것인지 
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
import os

from torch.nn.parallel import DistributedDataParallel as DDP

def example(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    # create local model
    model = nn.Linear(10, 10).to(rank)

    # 모든 model에 0번과 같은 파라미터를 갖도록
    for param in model.parameters():
        dist.broadcast(param, src=0)
    
    # define loss function
    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001)

    # forward
    outputs = model(torch.randn(20, 10).to(rank))
    labels = torch.randn(20, 10).to(rank)
    # backward
    loss = loss_fn(outputs, labels)
    loss.backward()

    for param in model.parameters():
        dist.all_reduce(param.grad, dist.ReduceOp.SUM)

    with torch.no_grad():
        for param in model.parameters():
            param.grad = param.grad / world_size

    first_param_grad = next(model.parameters()).grad
    print(f"Rank {rank} Gradient (first 5): {first_param_grad.flatten()[:5]}")
    # update
    optimizer.step()

    # Grad를 이용해서 parameter를 업데이트 했으므로 검증
    first_param_weight = next(model.parameters()).data
    print(f" Rank {rank} Weight (first 5): {first_param_weight.flatten()[:5]}")
    dist.destroy_process_group()
    
if __name__=="__main__":
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    world_size = 2
    mp.spawn(example, args=(world_size,),
            nprocs=world_size, join=True)

Overwriting train.py


In [ ]:
!python train.py